In [1]:
from env_vars import *

In [2]:
CORPUS_KEY = "en-tafisr-ibn-kathir"
CORPUS_NAME = "Tafsir Ibn Kathir (Abridged) - English"
ENCODER_ID = "enc_1"

In [5]:
import json
import os

import requests

API_KEY = os.getenv("VECTARA_AUTH_TOKEN", "zut_tECdwgW51oNpf1ZTvu56G7QtyrmBFvZ0upZ1MA")
CORPUS_KEY = "en-tafisr-ibn-kathir"
CORPUS_NAME = "Tafsir Ibn Kathir (Abridged) - English"
ENCODER_ID = "enc_1"

payload = {
    "key": CORPUS_KEY,
    "name": CORPUS_NAME,
    "description": "This corpus contains an English translation "
    "of the abridged Tafsir by Ibn Kathir, "
    "sourced from Quran.com.",
    "queries_are_answers": False,
    "documents_are_questions": False,
    "encoder_id": ENCODER_ID,
    "filter_attributes": [
        {
            "name": "ayah_id",
            "level": "part",
            "description": "The ayah ID is a unique identifier "
            "for each ayah in the Quran. It is "
            "formatted as surah_id:ayah_id, where "
            "surah_id represents the surah number "
            "and ayah_id represents the ayah number "
            "within that surah.",
            "indexed": True,
            "type": "text",
        }
    ],
    "custom_dimensions": [],
}

url = "https://api.vectara.io/v2/corpora"
headers = {
    "Content-Type": "application/json",
    "Accept": "application/json",
    "x-api-key": API_KEY,
}

response = requests.post(url, headers=headers, data=json.dumps(payload))

if response.status_code == 201:
    print("Corpus created successfully:", response.json())
else:
    print(f"Error creating corpus: {response.status_code}, " f"{response.text}")

In [3]:
with open("ayat_tafsirs.json") as f:
    ayat_tafsirs = json.load(f)
# format the data
surah_to_tafsir_by_ayah = {}
for ayah_id, tafsir in ayat_tafsirs.items():
    surah_id, ayah_id = ayah_id.split(":")
    if surah_id not in surah_to_tafsir_by_ayah:
        surah_to_tafsir_by_ayah[surah_id] = {}
    surah_to_tafsir_by_ayah[surah_id][ayah_id] = tafsir

In [4]:
from urllib.request import urlopen

url = "https://raw.githubusercontent.com/semarketir/quranjson/master/source/surah.json"

# Download the JSON data
response = urlopen(url)
surahs_metadata = json.loads(response.read().decode("utf-8"))

In [6]:
import requests
import json

url = f"https://api.vectara.io/v2/corpora/{CORPUS_KEY}/documents"
for (surah_id, ayahs), smetadata in zip(
    surah_to_tafsir_by_ayah.items(), surahs_metadata
):
    payload = json.dumps(
        {
            "id": surah_id,
            "type": "structured",
            "metadata": smetadata,
            "sections": [
                {
                    # "id": f"{surah_id}:{ayah_id}",
                    "text": tafsir,
                    "metadata": {"ayah_id": f"{surah_id}:{ayah_id}"},
                    #"context": "string",
                    #"custom_dimensions": {},
                }
                for ayah_id, tafsir in ayahs.items()
            ],
        }
    )
    headers = {
        "Content-Type": "application/json",
        "Accept": "application/json",
        "x-api-key": API_KEY,
    }
    response = requests.request("POST", url, headers=headers, data=payload)
    print(f"request sent for {surah_id}: {smetadata['titleAr']}")
    print(response.status_code)
    print(response.json())
    print("---")